# 06: 학습된 어댑터 평가

같은 실험의 평가 데이터와 학습된 어댑터와 토크나이저를 사용합니다. 기존 정규화, 중복 보존 완전 일치, 값 토큰 신뢰도 계산을 내부에 포함합니다. 기본 모델은 FP16으로 불러와 어댑터를 병합하므로 4비트 학습과 수치 차이가 있을 수 있습니다.


## 파라미터와 경로

상단의 `RUNTIME_PARAMS`와 `HYPER_PARAMS`를 수정하세요. 각 함수는 두 딕셔너리를 전역 설정으로 직접 참조합니다. 모든 파일 경로는 `RUNTIME_PARAMS["prefix_path"]` 기준입니다.


In [ ]:
# 모든 파일 경로는 prefix_path 기준입니다.
RUNTIME_PARAMS = {'prefix_path': '/drive/MyDrive/SiteBoard/06/',
 'model_name': 'Qwen/Qwen2.5-3B-Instruct',
 'adapter_path': 'models',
 'train_file': 'train.jsonl',
 'test_file': 'test.jsonl',
 'report_file': 'result/evaluation.json',
 'device': 'cuda',
 'model_revision': None}

HYPER_PARAMS = {
    # 문장 결합 증강 설정, 학습 입력의 조합을 다양화합니다.
    'augmentation': {
        'probability': 0.5,   # 각 학습 샘플에 문장 결합 증강을 적용할 확률입니다.
        'max_concat': 2,      # 한 번에 결합할 원본 샘플 수의 상한입니다.
        'delimiters': [       # 결합 문장 사이에 무작위로 넣을 구분자 후보입니다.
            ' ',
            '  ',
            ' / ',
            ' . ',
            '\n',
            ' , ',
            ' + ',
        ],
        'weighted': True,     # 희소 공종을 우선 선택하는 가중 파트너 샘플링을 켭니다.
        'weight_power': 0.7,  # 공종 역빈도에 적용할 지수로 희소 공종의 선택 비중을 조절합니다.
        'min_weight': 1.0,    # 샘플링 가중치의 하한입니다. 현재 1.0에서는 가중치가 균등해집니다.
    },

    # 학습 기본 설정, 반복 횟수와 최적화 조건을 정합니다.
    'training': {
        'epochs': 3,              # 전체 학습 데이터를 반복해서 학습할 횟수입니다.
        'batch_size': 2,          # 장치당 한 번의 학습 순전파에 넣을 샘플 수입니다.
        'grad_accum': 4,          # 지정한 미니배치 횟수만큼 기울기를 누적한 뒤 가중치를 갱신합니다.
        'learning_rate': 0.0002,  # 최적화기의 기본 학습률로 가중치 갱신 크기를 조절합니다.
        'max_seq_length': 1280,   # 학습과 검증에서 입력과 정답을 합친 토큰 길이의 상한입니다.
        'seed': 42,               # 난수 시드로 학습과 증강의 무작위 동작을 재현하도록 합니다.
    },

    # LoRA 어댑터 설정, 학습할 저차원 가중치의 구조를 정합니다.
    'lora': {
        'r': 16,                   # LoRA 저차원 행렬의 랭크로 학습 용량과 메모리 사용량을 조절합니다.
        'lora_alpha': 32,          # LoRA 갱신량에 적용할 배율의 기준값이며 r과 함께 스케일을 정합니다.
        'lora_dropout': 0.05,      # LoRA 경로의 입력을 일부 제외해 과적합을 줄이는 확률입니다.
        'task_type': 'CAUSAL_LM',  # 인과적 언어 모델 학습에 맞는 PEFT 동작을 선택합니다.
        'bias': 'none',            # 편향 파라미터의 학습 범위이며 none은 편향을 학습하지 않습니다.
        'target_modules': [        # LoRA를 부착할 모델 내부 선형 계층 이름입니다.
            'q_proj',
            'k_proj',
            'v_proj',
            'o_proj',
            'gate_proj',
            'up_proj',
            'down_proj',
        ],
    },

    # 학습 모델의 양자화 설정, 가중치 메모리 사용량을 줄입니다.
    'quantization': {
        'load_in_4bit': True,               # 기본 모델 가중치를 4비트로 불러와 GPU 메모리를 절약합니다.
        'bnb_4bit_quant_type': 'nf4',       # 4비트 가중치의 표현 방식이며 nf4 양자화를 사용합니다.
        'bnb_4bit_use_double_quant': True,  # 양자화 상수도 추가 양자화해 메모리 사용량을 줄입니다.
    },

    # 학습 모델 로딩 설정, 장치 배치와 로딩 자원을 정합니다.
    'model_loading': {
        'device_map': {'': 0},          # 모델 전체를 0번 GPU에 배치합니다. 빈 키는 모델 전체를 뜻합니다.
        'attn_implementation': 'sdpa',  # 어텐션 계산 구현을 선택하며 sdpa는 PyTorch SDPA를 사용합니다.
        'low_cpu_mem_usage': True,      # 모델 로딩 중 CPU 메모리의 최대 사용량을 줄입니다.
    },

    # 학습 토크나이저 설정, 입력 길이를 맞추는 방식을 정합니다.
    'tokenizer': {
        'padding_side': 'right',  # 패딩 토큰을 붙일 방향이며 right는 입력 오른쪽에 붙입니다.
    },

    # 양자화 모델의 학습 준비 설정입니다.
    'kbit_training': {
        'use_gradient_checkpointing': True,  # 역전파 때 중간값을 재계산해 활성값 저장 메모리를 줄입니다.
    },

    # 학습기 설정, 검증과 저장 주기 및 실행 자원을 정합니다.
    'trainer': {
        'per_device_eval_batch_size': 1,  # 학습 중 장치당 검증 배치 크기로 검증 메모리 사용량을 조절합니다.
        'lr_scheduler_type': 'cosine',    # 학습률 변화 방식이며 cosine은 코사인 곡선으로 감소시킵니다.
        'warmup_steps': 10,               # 초기 학습률을 점진적으로 높이는 최적화 스텝 수입니다.
        'logging_steps': 10,              # 학습 로그를 남기는 최적화 스텝 간격입니다.
        'save_strategy': 'epoch',         # 체크포인트 저장 시점이며 epoch는 각 에포크 종료 시 저장합니다.
        'eval_strategy': 'epoch',         # 검증 시점이며 epoch는 각 에포크 종료 시 검증합니다.
        'save_total_limit': 2,            # 유지할 체크포인트 수의 상한으로 저장 공간을 관리합니다.
        'fp16': True,                     # FP16 혼합 정밀도 학습으로 메모리와 연산 비용을 줄입니다.
        'dataloader_num_workers': 2,      # 데이터 로딩에 사용할 작업 프로세스 수입니다.
        'dataloader_pin_memory': True,    # 고정 CPU 메모리로 배치를 준비해 GPU 전송을 돕습니다.
        'report_to': 'none',              # 외부 실험 추적 서비스이며 none은 외부 전송을 끕니다.
        'remove_unused_columns': False,   # 사용하지 않는 데이터 열의 자동 제거 여부를 정합니다.
    },

    # 배치 구성 설정, 샘플 길이와 텐서 형식을 맞춥니다.
    'collator': {
        'pad_to_multiple_of': 8,  # 배치 길이를 지정한 배수로 맞춰 GPU 연산 효율을 돕습니다.
        'return_tensors': 'pt',   # 배치의 반환 형식이며 pt는 PyTorch 텐서를 뜻합니다.
        'padding': True,          # 같은 배치의 샘플 길이를 패딩으로 맞춥니다.
    },

    # 평가 노트북의 추론 설정, 배치 크기와 토큰 상한을 정합니다.
    'evaluation': {
        'batch_size': 2,           # 평가 시 한 번에 추론할 샘플 수로 메모리 사용량을 조절합니다.
        'max_new_tokens': 512,     # 샘플당 새로 생성할 응답 토큰 수의 상한입니다.
        'max_input_tokens': 1024,  # 평가 입력을 토큰화할 때 적용하는 길이 상한입니다.
    },
}


## 1. 의존성 설치


In [ ]:
# Colab GPU 런타임에서 실행하세요. 설치 후 필요하면 런타임을 재시작하세요.
!pip install "torch>=2.2.0" "transformers>=4.46.0,<5" "peft>=0.13.0,<0.18" "accelerate>=0.29.0,<2" "bitsandbytes>=0.43.0,<1" "tqdm>=4.66.0"


In [ ]:
import json
import os
import platform
import re
from collections import Counter
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch


## 2. 실행 정보, 모델 로딩


In [ ]:
def runtime_path(field):
    return str(Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field])


def collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "packages": packages,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or RUNTIME_PARAMS.get("model_revision"),
    }


def read_jsonl(path):
    path = Path(path)
    records = []
    with path.open("r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON 문법 오류: {path}:{line_number}") from exc
            messages = record.get("messages") if isinstance(record, dict) else None
            if not isinstance(messages, list) or len(messages) != 3:
                raise ValueError(f"system, user, assistant 메시지 3개가 필요합니다: {path}:{line_number}")
            for message, role in zip(messages, ("system", "user", "assistant")):
                if (not isinstance(message, dict) or message.get("role") != role
                        or not isinstance(message.get("content"), str)):
                    raise ValueError(f"메시지 역할과 문자열 content를 확인하세요: {path}:{line_number}")
            records.append(record)
    if not records:
        raise ValueError(f"데이터가 비어 있습니다: {path}")
    return records


def load_model():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    adapter = runtime_path("adapter_path")
    if not Path(adapter).is_dir():
        raise FileNotFoundError(f"어댑터가 없습니다: {adapter}. 먼저 학습하거나 RUNTIME_PARAMS의 adapter_path에 보유 모델을 지정하세요.")
    device = RUNTIME_PARAMS.get("device", "cuda")
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Colab에서 CUDA GPU 런타임을 선택하세요.")
    if device == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS를 사용할 수 없습니다.")
    dtype = torch.float32 if device == "cpu" else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(adapter, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS.get("model_revision"), torch_dtype=dtype, device_map={"": device},
        attn_implementation="sdpa", low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(model, adapter).merge_and_unload()
    model.eval()
    name = torch.cuda.get_device_name(0) if device == "cuda" else device
    return model, tokenizer, name


## 3. 평가 지표


In [ ]:
def parse_and_normalize(json_str: str):
    """JSON 문자열을 파싱하여 정규화된 튜플 리스트로 반환합니다."""
    cleaned = re.sub(r"<think>.*?</think>", "", json_str, flags=re.DOTALL).strip()
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return [(str(item.get("workType", "")).strip(),
                     str(item.get("spec", "")).strip(),
                     round(float(item.get("quantity", 0)), 4)) for item in data if isinstance(item, dict)]
    except Exception:
        pass
    return None


def extract_values_with_confidence(pred_text: str, token_spans: List[Tuple[int, int, str, float]]):
    """JSON 문법 토큰을 제외하고, 실제 workType, spec, quantity 값 구간의 토큰 확률만 추출합니다."""
    item_pattern = re.compile(r"\{[^{}]*\}")
    work_type_pat = re.compile(r"\"workType\"\s*:\s*\"([^\"]*)\"")
    spec_pat = re.compile(r"\"spec\"\s*:\s*\"([^\"]*)\"")
    qty_pat = re.compile(r"\"quantity\"\s*:\s*([0-9.]+)")

    results = []
    all_val_probs = []

    def get_probs_in_span(span_start: int, span_end: int) -> List[float]:
        collected = []
        for t_s, t_e, _, t_prob in token_spans:
            if max(span_start, t_s) < min(span_end, t_e):
                collected.append(t_prob)
        return collected

    for m_item in item_pattern.finditer(pred_text):
        item_str = m_item.group(0)
        item_start = m_item.start()

        wt_m = work_type_pat.search(item_str)
        spec_m = spec_pat.search(item_str)
        qty_m = qty_pat.search(item_str)

        item_dict: Dict[str, Any] = {}

        if wt_m:
            val = wt_m.group(1).strip()
            span = (item_start + wt_m.start(1), item_start + wt_m.end(1))
            val_probs = get_probs_in_span(span[0], span[1])
            all_val_probs.extend(val_probs)
            item_dict["workType"] = val
            item_dict["workType_confidence"] = round(float(sum(val_probs) / len(val_probs)), 4) if val_probs else 1.0

        if spec_m:
            val = spec_m.group(1).strip()
            span = (item_start + spec_m.start(1), item_start + spec_m.end(1))
            val_probs = get_probs_in_span(span[0], span[1])
            all_val_probs.extend(val_probs)
            item_dict["spec"] = val
            item_dict["spec_confidence"] = round(float(sum(val_probs) / len(val_probs)), 4) if val_probs else 1.0

        if qty_m:
            val = qty_m.group(1).strip()
            span = (item_start + qty_m.start(1), item_start + qty_m.end(1))
            val_probs = get_probs_in_span(span[0], span[1])
            all_val_probs.extend(val_probs)
            try:
                item_dict["quantity"] = float(val)
            except Exception:
                item_dict["quantity"] = val
            item_dict["quantity_confidence"] = round(float(sum(val_probs) / len(val_probs)), 4) if val_probs else 1.0

        if item_dict:
            results.append(item_dict)

    min_val_conf = float(min(all_val_probs)) if all_val_probs else 0.0
    avg_val_conf = float(sum(all_val_probs) / len(all_val_probs)) if all_val_probs else 0.0
    return results, round(avg_val_conf, 4), round(min_val_conf, 4)


## 4. 평가, 개별 예측, 결과 저장


In [ ]:
def evaluate():
    import torch
    import torch.nn.functional as F
    from tqdm import tqdm

    if Path(runtime_path("report_file")).exists():
        raise FileExistsError("기존 평가 결과가 있습니다. 새 실험 ID를 사용하세요.")
    test_samples = read_jsonl(runtime_path("test_file"))
    MODEL_NAME = RUNTIME_PARAMS["model_name"]
    ADAPTER_PATH = runtime_path("adapter_path")
    TEST_PATH = runtime_path("test_file")
    OUTPUT_REPORT = runtime_path("report_file")
    MAX_NEW_TOKENS = HYPER_PARAMS["evaluation"]["max_new_tokens"]
    EVAL_BATCH_SIZE = HYPER_PARAMS["evaluation"]["batch_size"]
    model, tokenizer, device_name = load_model()
    eos_ids = [tokenizer.eos_token_id]
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id not in eos_ids:
        eos_ids.append(im_end_id)

    total_samples = len(test_samples)
    print(f"\n[3/3] 총 {total_samples}건 배치 고속 추론 및 평가 시작 (Batch={EVAL_BATCH_SIZE})...\n")

    case_results = []
    correct_count = 0

    # 배치 단위 순회
    for i in tqdm(range(0, total_samples, EVAL_BATCH_SIZE), desc="T4 Batched Inference"):
        batch_samples = test_samples[i:i + EVAL_BATCH_SIZE]
        
        # 1) 배치 프롬프트 구성
        batch_prompts = []
        for sample in batch_samples:
            messages = sample["messages"]
            system_msg = next((m["content"] for m in messages if m["role"] == "system"), "")
            user_msg = next((m["content"] for m in messages if m["role"] == "user"), "")
            
            eval_messages = []
            if system_msg:
                eval_messages.append({"role": "system", "content": system_msg})
            eval_messages.append({"role": "user", "content": user_msg})
            
            prompt = tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
            batch_prompts.append(prompt)

        # 2) 배치 일괄 토큰화 및 GPU 적재
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=HYPER_PARAMS["evaluation"]["max_input_tokens"]
        ).to(model.device)

        prompt_len = inputs["input_ids"].shape[1]

        # 3) T4 FP16 배치 추론 (inference_mode + scores 반환)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=eos_ids,
                return_dict_in_generate=True,
                output_scores=True
            )

        # 4) 출력 슬라이싱 및 배치 일괄 디코딩
        gen_tokens_batch = outputs.sequences[:, prompt_len:]
        pred_strings = tokenizer.batch_decode(gen_tokens_batch, skip_special_tokens=True)

        # 5) 개별 샘플 검증, 값 단위(Value-level) 신뢰도 계산 및 결과 집계
        for j, sample in enumerate(batch_samples):
            case_id = i + j + 1
            pred_output_str = pred_strings[j].strip()
            user_msg = next((m["content"] for m in sample["messages"] if m["role"] == "user"), "")
            true_output_str = next((m["content"] for m in sample["messages"] if m["role"] == "assistant"), "")

            # 토큰별 Softmax 확률 및 문자열 Span 매핑
            token_ids = gen_tokens_batch[j]
            token_spans = []
            cur_pos = 0

            for step_idx, step_logits in enumerate(outputs.scores):
                if step_idx >= len(token_ids):
                    break
                tid = token_ids[step_idx].item()
                if tid == tokenizer.pad_token_id:
                    break
                
                prob = F.softmax(step_logits[j], dim=-1)[tid].item()
                t_str = tokenizer.decode([tid])
                t_start = cur_pos
                cur_pos += len(t_str)
                t_end = cur_pos
                token_spans.append((t_start, t_end, t_str, prob))

                if tid in eos_ids:
                    break

            # 실제 값(Value: workType, spec, quantity) 구간의 토큰 신뢰도만 추출
            val_predictions, val_avg_conf, val_min_conf = extract_values_with_confidence(pred_output_str, token_spans)

            # 정답 및 예측 비교 (Record-Level Multiset Exact Match)
            norm_pred = parse_and_normalize(pred_output_str)
            norm_true = parse_and_normalize(true_output_str)
            is_match = (norm_pred is not None and norm_true is not None and Counter(norm_pred) == Counter(norm_true))

            if is_match:
                correct_count += 1

            case_results.append({
                "case_id": case_id,
                "input_user": user_msg,
                "ground_truth_raw": true_output_str,
                "prediction_raw": pred_output_str,
                "prediction": val_predictions,
                "is_exact_match": is_match,
                "value_avg_confidence": val_avg_conf,
                "value_min_confidence": val_min_conf
            })

        # 배치별 어휘 점수를 다음 생성 전에 해제합니다.
        del outputs, inputs, gen_tokens_batch, token_ids
        step_logits = None

    accuracy = (correct_count / total_samples) * 100.0 if total_samples else 0.0
    avg_val_conf = (sum(c["value_avg_confidence"] for c in case_results) / total_samples) if total_samples else 0.0
    correct_cases = [c for c in case_results if c["is_exact_match"]]
    incorrect_cases = [c for c in case_results if not c["is_exact_match"]]
    correct_avg_val_conf = (sum(c["value_avg_confidence"] for c in correct_cases) / len(correct_cases)) if correct_cases else 0.0
    incorrect_avg_val_conf = (sum(c["value_avg_confidence"] for c in incorrect_cases) / len(incorrect_cases)) if incorrect_cases else 0.0

    print("\n" + "=" * 70)
    print(f"📊 Strict Exact Match Accuracy        : {accuracy:.1f}% ({correct_count}/{total_samples})")
    print(f"📈 Overall Value Avg Confidence       : {avg_val_conf * 100:.2f}%")
    print(f"   • Correct Cases Value Avg Conf    : {correct_avg_val_conf * 100:.2f}%")
    print(f"   • Incorrect Cases Value Avg Conf  : {incorrect_avg_val_conf * 100:.2f}%")
    print("=" * 70)

    report = {
        "model": MODEL_NAME,
        "adapter": ADAPTER_PATH,
        "device": device_name,
        "dtype": str(model.dtype),
        "test_file": TEST_PATH,
        "runtime_params": deepcopy(RUNTIME_PARAMS),
        "hyper_params": deepcopy(HYPER_PARAMS),
        "batch_size": EVAL_BATCH_SIZE,
        "accuracy": accuracy,
        "total": total_samples,
        "correct": correct_count,
        "value_avg_confidence": round(avg_val_conf, 4),
        "correct_value_avg_confidence": round(correct_avg_val_conf, 4),
        "incorrect_value_avg_confidence": round(incorrect_avg_val_conf, 4),
        "execution": collect_execution_metadata(model),
        "cases": case_results
    }

    report_dir = os.path.dirname(OUTPUT_REPORT)
    if report_dir:
        os.makedirs(report_dir, exist_ok=True)
    with open(OUTPUT_REPORT, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"\n📁 평가 리포트 저장 완료: {OUTPUT_REPORT}")
    return report


## 5. 실행

GPU 메모리가 부족하면 평가 배치를 낮추고 새 런타임에서 실행하세요. 입력과 출력 토큰 상한은 평가 조건이므로 함께 확인하세요. 기존 평가 결과는 덮어쓰지 않습니다.


In [ ]:
report = evaluate()
